Smart Parking Management System

Part 1 — Dataset preparation

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),

(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

In [0]:
Part 2 — Assessment questions

1

In [0]:
%sql
SELECT *
FROM parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


In [0]:
%sql
SELECT session_id, vehicle_type, parking_fee
FROM parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


2

In [0]:
%sql
SELECT *
FROM parking_sessions_source
WHERE status = 'Completed'
AND parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


3

In [0]:
%sql
SELECT
    vehicle_type,
    SUM(parking_fee) AS total_parking_fee,
    COUNT(*) AS total_sessions
FROM parking_sessions_source
WHERE status = 'Completed'
GROUP BY vehicle_type;

vehicle_type,total_parking_fee,total_sessions
Car,1750,8
Bike,160,4
SUV,540,2


4

In [0]:
from pyspark.sql.functions import col, when

df = spark.table("parking_sessions_source")

df_result = df.withColumn(
    "fee_category",
    when(col("parking_fee") >= 200, "High")
    .otherwise("Low")
)

display(df_result)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


Section B — Delta Tables and CRUD (20 marks)

5

In [0]:
%sql
CREATE OR REPLACE TABLE parking_summary
USING DELTA
AS
SELECT * FROM parking_sessions_source
WHERE status ="Completed";

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM parking_summary;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed


6

In [0]:
%sql
INSERT INTO parking_sessions_work
VALUES(123,"Z01","Car", 3,150,"Active");

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT * FROM parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


7

In [0]:
%sql
UPDATE parking_sessions_work
SET
    status="Completed",
    parking_fee=280
WHERE session_id=104;
    


num_affected_rows
1


In [0]:
%sql
SELECT * FROM parking_sessions_work
WHERE session_id=104;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed


8

In [0]:
%sql
DELETE FROM parking_sessions_work
WHERE status = 'Cancelled';

num_affected_rows
2


In [0]:
%sql
SELECT *
FROM parking_sessions_work;

Section C — MERGE and Delta Operations (20 marks)

9

In [0]:
%sql
SELECT *
FROM incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
SELECT *
FROM incoming_parking_sessions i
INNER JOIN parking_sessions_work w
ON i.session_id = w.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed,102,Z02,Bike,2,40,Completed
104,Z01,SUV,5,300,Completed,104,Z01,SUV,4,280,Completed


10

In [0]:
%sql
MERGE INTO parking_sessions_work AS target
USING incoming_parking_sessions AS source
ON target.session_id = source.session_id

WHEN MATCHED THEN
UPDATE SET
    target.zone_id = source.zone_id,
    target.vehicle_type = source.vehicle_type,
    target.duration_hours = source.duration_hours,
    target.parking_fee = source.parking_fee,
    target.status = source.status

WHEN NOT MATCHED THEN
INSERT (
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
)
VALUES (
    source.session_id,
    source.zone_id,
    source.vehicle_type,
    source.duration_hours,
    source.parking_fee,
    source.status
);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
SELECT *
FROM parking_sessions_work
ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,3,60,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,5,300,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


11

In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T05:08:02.000Z,76710758477959,dsairaghul9@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1485798670704334),a8ae459d-39e8-4ceb-a888-05312c4e83b4,1106-042208-62e2xx6q-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9875, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T05:08:00.000Z,76710758477959,dsairaghul9@gmail.com,MERGE,"Map(predicate -> [""(session_id#14330 = session_id#14312)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1485798670704334),a8ae459d-39e8-4ceb-a888-05312c4e83b4,1106-042208-62e2xx6q-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7597, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3613, materializeSourceTimeMs -> 349, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1092, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2088)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:58:53.000Z,76710758477959,dsairaghul9@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1485798670704334),1b9f44c6-40ee-49c9-a36d-d581c901c074,1106-042208-62e2xx6q-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:58:51.000Z,76710758477959,dsairaghul9@gmail.com,DELETE,"Map(predicate -> [""(status#13614 = Cancelled)""])",null,List(1485798670704334),1b9f44c6-40ee-49c9-a36d-d581c901c074,1106-042208-62e2xx6q-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1091, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 618, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 472)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:56:58.000Z,76710758477959,dsairaghul9@gmail.com,UPDATE,"Map(predicate -> [""(session_id#13042 = 104)""])",null,List(1485798670704334),1ec78098-d3ab-4f8d-a759-c3e7c46b24f1,1106-042208-62e2xx6q-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3758, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1816, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 1911)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:51:51.000Z,76710758477959,dsairaghul9@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1485798670704334),3bf45e4d-5efa-41fc-9427-3000dbe512b9,1106-042208-62e2xx6q-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutpu

12

In [0]:
%sql
SELECT * FROM parking_sessions_work
VERSION AS OF 4;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
SELECT *
FROM parking_sessions_work VERSION AS OF 4
WHERE session_id = 102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,2,40,Completed


In [0]:
%sql
SELECT *
FROM parking_sessions_work 
WHERE session_id = 102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


Section D — Tables, Views and Optimization (20 marks)

13

In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT *
FROM parking_sessions_work
WHERE status = 'Completed';

14

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW zone_z01_view AS
SELECT *
FROM parking_sessions_work
WHERE zone_id = 'Z01';

In [0]:
%sql
SELECT *
FROM zone_z01_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_view AS
SELECT *
FROM parking_sessions_work
WHERE parking_fee > 200;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7349824085214280>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_view AS\nSELECT *\nFROM parking_sessions_work\nWHERE parking_fee > 200;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cel

In [0]:
%sql
SELECT *
FROM global_temp.high_fee_view;

15

In [0]:
%sql
SELECT
    w.zone_id,
    z.zone_name,
    z.city,
    w.vehicle_type,
    w.parking_fee
FROM parking_sessions_work w
INNER JOIN parking_zones z
ON w.zone_id = z.zone_id;

zone_id,zone_name,city,vehicle_type,parking_fee
Z01,Central Plaza,Hyderabad,Car,150
Z03,City Mall,Mumbai,Car,250
Z04,Airport Parking,Delhi,Bike,20
Z02,Metro Parking,Bengaluru,Car,300
Z05,Tech Park,Pune,Car,150
Z01,Central Plaza,Hyderabad,Bike,80
Z04,Airport Parking,Delhi,SUV,300
Z02,Metro Parking,Bengaluru,Car,100
Z03,City Mall,Mumbai,Car,350
Z01,Central Plaza,Hyderabad,SUV,180


16

In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,07e46702-54f4-421b-89f7-cdb56567cc4a,workspace.default.parking_sessions_work,null,,2026-11-06T04:23:59.513Z,2026-11-06T05:08:02.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,07e46702-54f4-421b-89f7-cdb56567cc4a,workspace.default.parking_sessions_work,null,,2026-11-06T04:23:59.513Z,2026-11-06T05:08:02.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,07e46702-54f4-421b-89f7-cdb56567cc4a,workspace.default.parking_sessions_work,null,,2026-11-06T04:23:59.513Z,2026-11-06T05:08:02.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


OPTIMIZE combines smaller data files into larger files to improve query performance.

A small dataset might show no significant change because:
- It contains very few records.
- It already has a small number of files.
- There is no major file fragmentation.

Section E — Structured Streaming and DLT (20 marks)

17

In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema


In [0]:
%sql
USE CATALOG workspace;
    
USE SCHEMA  default;
    
SHOW VOLUMES;
    


database,volume_name
default,parking_checkpoints


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.parking_checkpoints;

In [0]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

query17 = (
    rate_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/parking_checkpoints/q17_delta"
    )
    .toTable("workspace.default.parking_rate_output")
)


query17.awaitTermination()


display(spark.table("workspace.default.parking_rate_output"))

timestamp,value
2026-11-06T06:42:55.970Z,464
2026-11-06T06:42:57.570Z,472
2026-11-06T06:42:59.170Z,480
2026-11-06T06:43:00.770Z,488
2026-11-06T06:43:02.370Z,496
2026-11-06T06:43:03.970Z,504
2026-11-06T06:43:05.570Z,512
2026-11-06T06:43:07.170Z,520
2026-11-06T06:43:08.770Z,528
2026-11-06T06:43:10.370Z,536


18

In [0]:
from pyspark.sql.functions import col

amount_df = (
    rate_df
    .withColumn("amount", col("value"))
    .filter(col("amount") > 300)
)

query18 = (
    amount_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/parking_checkpoints/q18_delta"
    )
    .toTable("workspace.default.parking_amount_output")
)


query18.awaitTermination()


display(spark.table("workspace.default.parking_amount_output"))

timestamp,value,amount
2026-11-06T06:47:15.328Z,304,304
2026-11-06T06:47:16.928Z,312,312
2026-11-06T06:47:18.528Z,320,320
2026-11-06T06:47:20.128Z,328,328
2026-11-06T06:47:21.728Z,336,336
2026-11-06T06:47:23.328Z,344,344
2026-11-06T06:47:24.928Z,352,352
2026-11-06T06:47:26.528Z,360,360
2026-11-06T06:47:28.128Z,368,368
2026-11-06T06:47:29.728Z,376,376


19

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_parking_sessions
AS
SELECT *
FROM STREAM(parking_sessions_source);

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE silver_parking_sessions
AS
SELECT *
FROM STREAM(bronze_parking_sessions)
WHERE status = 'Completed';

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW gold_parking_summary
AS
SELECT
    zone_id,
    COUNT(*) AS total_completed_sessions,
    SUM(parking_fee) AS total_parking_revenue
FROM silver_parking_sessions
GROUP BY zone_id;

result
The operation was successfully executed.


In [0]:
%sql
SELECT *
FROM gold_parking_summary;

zone_id,total_completed_sessions,total_parking_revenue
Z03,3,620
Z01,3,410
Z02,4,800
Z04,2,270
Z05,2,350
